# Market Price Model — Marine Fishing AI (Kerala)

**⚠️ This trains on `data/mock/mock_monthly_prices.csv` — SYNTHETIC data, not real market history.**

Real price data currently covers exactly one week (24-30 Mar 2025), which isn't enough to show any seasonal
pattern. This notebook trains against a documented synthetic dataset (real anchor prices from that one week,
scaled by an assumed seasonal pattern tied to Kerala's known trawling-ban timing — see
`src/ml/generate_mock_price_data.py` for the exact assumption). Treat everything below as a **pipeline/prototype
validation**, not a real market-price predictor. When real multi-month price data exists, re-run this notebook
against that instead — the code doesn't need to change, only the input file.

Per the project's data-pipeline design: all data prep/cleaning logic lives in the repo (`src/ml/`), never
duplicated here. This notebook only does: load → train → evaluate → export.

## 1. Clone the repo and install dependencies

In [ ]:
# Replace with your actual GitHub repo URL / branch if different
!git clone -b feature/model-training https://github.com/vishaldharsan27-ux/Marine_AI.git repo
%cd repo
!pip install -q -r requirements.txt scikit-learn joblib

## 2. Load the mock price data

If `data/mock/mock_monthly_prices.csv` isn't in the cloned branch yet, regenerate it directly:
`!python -m src.ml.generate_mock_price_data`

In [ ]:
import pandas as pd

df = pd.read_csv("data/mock/mock_monthly_prices.csv")
assert df["is_synthetic"].all(), "Expected every row to be flagged synthetic - stop if this fails"
print(f"{len(df)} rows, {df['species_key'].nunique()} species, months {sorted(df['month'].unique())}")
df.head(12)

## 3. Feature engineering

Features: species (categorical) + month (1-12, encoded cyclically so December and January are recognized as
adjacent rather than far apart on a linear scale). Target: `price_inr_per_kg`.

In [ ]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

species_enc = LabelEncoder().fit(df["species_key"])
df["species_id"] = species_enc.transform(df["species_key"])
df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)

FEATURES = ["species_id", "month_sin", "month_cos"]
TARGET = "price_inr_per_kg"
df[FEATURES + [TARGET]].head()

## 4. Train / holdout split and fit

Holding out 3 months (Jun, Sep, Dec — one from each seasonal regime: ban spike, post-monsoon dip, baseline) so
the evaluation actually checks whether the model learned the shape of the season, not just memorized every row.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

HOLDOUT_MONTHS = [6, 9, 12]
train_df = df[~df["month"].isin(HOLDOUT_MONTHS)]
test_df = df[df["month"].isin(HOLDOUT_MONTHS)]

model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2)
model.fit(train_df[FEATURES], train_df[TARGET])

preds = model.predict(test_df[FEATURES])
mae = mean_absolute_error(test_df[TARGET], preds)
print(f"Holdout MAE (months {HOLDOUT_MONTHS}): Rs.{mae:.1f}/kg")
print(f"(for scale: holdout prices range Rs.{test_df[TARGET].min():.0f} to Rs.{test_df[TARGET].max():.0f}/kg)")

## 5. Inspect predictions across the full year, per species

Sanity check: does the model actually reproduce the ban-season spike / post-monsoon dip pattern, or did it just
learn a flat per-species average?

In [ ]:
full_preds = model.predict(df[FEATURES])
df["predicted_price"] = full_preds

pivot = df.pivot_table(index="month_name", columns="species_key", values="predicted_price", sort=False)
pivot = pivot.reindex(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"])
pivot.round(1)

## 6. Retrain on all months and export the artifact

Saved locally in the Colab runtime. Download it or push it back to a branch (e.g. via a GitHub PAT stored as a
Colab secret) to get it into `src/ml/models/` in the actual repo — don't hand-copy weights out of the notebook.

In [ ]:
import joblib
from pathlib import Path

final_model = RandomForestRegressor(n_estimators=300, random_state=42, min_samples_leaf=2)
final_model.fit(df[FEATURES], df[TARGET])

Path("src/ml/models").mkdir(parents=True, exist_ok=True)
joblib.dump({
    "model": final_model,
    "species_encoder": species_enc,
    "features": FEATURES,
    "trained_on": "SYNTHETIC mock data - data/mock/mock_monthly_prices.csv",
}, "src/ml/models/market_price_model_MOCK.joblib")

print("Saved to src/ml/models/market_price_model_MOCK.joblib")
print("NOTE the _MOCK suffix is intentional - do not rename it away until this is retrained on real data.")